In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import nltk
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from string import punctuation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn import metrics
from xgboost import XGBRegressor
from pymystem3 import Mystem
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка nltk
nltk.data.path.append("C:/Users/glebt/AppData/Roaming/nltk_data")
nltk.data.path.append("C:/nltk_data")
nltk.download('punkt')
nltk.download('stopwords')

# Глобальные ресурсы
stop_words = set(stopwords.words("russian"))
custom_stop_words = {
    'обязанность', 'требование', 'условие', 'график', 'работа', 'компания',
    'оформление', 'тк', 'рф', 'заработный', 'плата', 'опыт', 'образование',
    'сотрудник', 'вакансия', 'предлагать', 'развитие', 'карьерный', 'рост',
    'официальный', 'трудоустройство', 'дружный', 'коллектив'
}
stop_words.update(custom_stop_words)
punct = set(punctuation)
stemmer = SnowballStemmer("russian")

# HTML очистка
def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator=' ', strip=True)

# Предобработка текста
def preprocess_text(text):
    local_mystem = Mystem()
    lemmas = [token.lower() for token in local_mystem.lemmatize(text)
              if token.strip() and token.isalpha()]
        
    cleaned_lemmas = [lemma for lemma in lemmas if lemma not in stop_words and lemma not in punct]
        
    # Сразу возвращаем очищенные леммы
    return ' '.join(cleaned_lemmas)

# Параллельная обработка
def parallel_process_texts(texts, max_workers=6):
    results = [None] * len(texts)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(preprocess_text, text): idx for idx, text in enumerate(texts)}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Обработка текстов"):
            idx = futures[future]
            results[idx] = future.result()
    return results

# Загрузка данных
df_train = pd.read_csv(r"C:\Users\glebt\Desktop\train_regr.csv")
df_test = pd.read_csv(r"C:\Users\glebt\Desktop\test_regr.csv")

# Очистка
df_train = df_train.dropna(subset=['description', 'salary_currency', 'salary_from','name']).drop_duplicates(subset=['description'])
df_test = df_test.dropna(subset=['description', 'salary_currency', 'salary_from','name']).drop_duplicates(subset=['description'])

# HTML очистка
df_train['clean_description'] = df_train['description'].apply(clean_html).fillna('').astype(str)
df_test['clean_description'] = df_test['description'].apply(clean_html).fillna('').astype(str)

# Предобработка
df_train['processed_text'] = parallel_process_texts(df_train['clean_description'].tolist())
df_test['processed_text'] = parallel_process_texts(df_test['clean_description'].tolist())

# Объединение и кодирование категорий
df_train['is_train'] = 1
df_test['is_train'] = 0
if 'salary_from' not in df_test.columns:
    df_test['salary_from'] = np.nan
df_all = pd.concat([df_train, df_test], ignore_index=True)

# 1. One-Hot Encoding для name (без предварительной очистки)
name_ohe = pd.get_dummies(df_all['name'], prefix='name')

# 2. Создание признаков seniority
def get_seniority(name):
    name_lower = str(name).lower()
    if any(word in name_lower for word in ['руководитель', 'начальник', 'директор', 'head']): return 'head'
    if any(word in name_lower for word in ['старший', 'senior', 'ведущий', 'lead']): return 'senior'
    if any(word in name_lower for word in ['младший', 'junior', 'стажер', 'ассистент', 'помощник', 'ученик']): return 'junior'
    return 'middle'

df_all['seniority'] = df_all['name'].apply(get_seniority)

# 2. Бинарные признаки из description
df_all['req_higher_edu'] = df_all['description'].str.contains('высшее образование', case=False, na=False).astype(int)
df_all['req_experience'] = df_all['description'].str.contains('опыт работы', case=False, na=False).astype(int)

# 3. One-Hot Encoding для категориальных признаков
categorical_features = df_all[["area_name", "salary_currency", "seniority"]]
df_all_dummies = pd.get_dummies(categorical_features, drop_first=True)

# 4. Числовые признаки
numerical_features = df_all[['req_higher_edu', 'req_experience']]

# 5. TF-IDF для processed_text
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.4
)
tfidf_matrix = tfidf.fit_transform(df_all['processed_text'])

# 6. Объединение всех признаков
X_all_sparse = hstack([
    csr_matrix(df_all_dummies.values),
    csr_matrix(numerical_features.values),
    tfidf_matrix
])

# 7. Разделение на train/test
train_mask = df_all['is_train'] == 1
test_mask = df_all['is_train'] == 0

# Преобразуем в numpy array
train_idx = np.where(train_mask.to_numpy())[0]
test_idx = np.where(test_mask.to_numpy())[0]

X_train = X_all_sparse[train_idx]
X_test = X_all_sparse[test_idx]
y_train = df_all.loc[train_mask, "salary_from"]
y_test = df_all.loc[test_mask, "salary_from"]

# 8. Логарифмирование и масштабирование
y_train_log = np.log1p(y_train)
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучение модели
param_dist = {
    'n_estimators': stats.randint(100, 1000),         # Количество деревьев
    'max_depth': stats.randint(3, 5),                # Максимальная глубина дерева
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],    # Скорость обучения
    'subsample': stats.uniform(0.6, 0.4),             # Доля строк для обучения каждого дерева (от 0.6 до 1.0)
    'colsample_bytree': stats.uniform(0.6, 0.4)       # Доля признаков для обучения каждого дерева (от 0.6 до 1.0)
}
model = XGBRegressor(
    random_state=42,    
    reg_alpha=0.1,        
    reg_lambda=0.1
    )
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    scoring='neg_mean_absolute_error', # Мы минимизируем отрицательную MAE (т.е. максимизируем MAE)
    verbose=2,
    random_state=42,
    n_jobs=1 
)
random_search.fit(X_train_scaled, y_train_log)
best_model = random_search.best_estimator_

# Прогноз
y_pred_log = best_model.predict(X_test_scaled)
y_pred_st = np.expm1(y_pred_log)
# Метрики
y_test_valid = y_test.dropna()
y_pred_metric = pd.Series(y_pred_st, index=y_test.index)[y_test_valid.index]

MAE_xgb = metrics.mean_absolute_error(y_test_valid, y_pred_metric)
RMSE_xgb = metrics.mean_squared_error(y_test_valid, y_pred_metric) ** 0.5
r2_xgb = metrics.r2_score(y_test_valid, y_pred_metric)

print(f"xgb модель: R² = {round(r2_xgb, 3)}, MAE = {round(MAE_xgb, 3)}, RMSE = {round(RMSE_xgb, 3)}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\glebt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\glebt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Обработка текстов: 100%|██████████| 10419/10419 [24:47<00:00,  7.00it/s]


Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.2, max_depth=3, n_estimators=206, subsample=0.9118764001091078; total time=  32.2s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.2, max_depth=3, n_estimators=206, subsample=0.9118764001091078; total time=  32.7s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.2, max_depth=3, n_estimators=206, subsample=0.9118764001091078; total time=  33.0s
[CV] END colsample_bytree=0.8387400631785948, learning_rate=0.05, max_depth=3, n_estimators=314, subsample=0.6232334448672797; total time=  51.9s
[CV] END colsample_bytree=0.8387400631785948, learning_rate=0.05, max_depth=3, n_estimators=314, subsample=0.6232334448672797; total time=  51.4s
[CV] END colsample_bytree=0.8387400631785948, learning_rate=0.05, max_depth=3, n_estimators=314, subsample=0.6232334448672797; total time=  51.4s
[CV] END colsample_bytree=0.9464704583099741, learning_rate=0.15, max_